# Benchmarks for Sec. 5: deformation-stability diagnostics

Small-system smoke tests for the three numerical layers used by Sec. 5:

1. exact-cage compatibility under a declared perturbation alphabet;
2. cage-conditioning singular values;
3. thermal-activity margins along a one-parameter path.

The production evidence remains in the spin-1 XY and square-QDM notebooks. This notebook is deliberately cheap enough for CI or a laptop.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from helpers import set_revtex_matplotlib_style, save_prx_figure
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageSearchConfig,
    CageSearcher,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_jacobian_conditioning_from_hamiltonian,
    operator_coefficient_compatibility,
    thermal_activity_margin_from_samples,
)
from qlinks.models import (
    SpinOneXYChainModel,
    SquareQDMModel,
    spin_one_xy_periodic_range_couplings,
    spin_one_xy_scar_tower_states,
)

TOL = 1.0e-10
set_revtex_matplotlib_style(base_font_size=9, prefer_tex=False)
DATA_DIR = REPO_ROOT / "experimental" / "data" / "deformation_stability_benchmarks"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Spin-1 XY: preserving versus obstructed perturbations

In [ ]:
L = 6
TOTAL_SZ = -2
base_model = SpinOneXYChainModel(
    length=L,
    boundary_condition="periodic",
    j_xy=2.0,
    d_z=0.63,
    total_sz=TOTAL_SZ,
    extra_xy_couplings=spin_one_xy_periodic_range_couplings(
        length=L,
        distance=3,
        coefficient=0.2,
    ),
)
base = base_model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
configs = basis_configs_from_build_result(base)
states, labels = spin_one_xy_scar_tower_states(
    basis_configs=configs,
    length=L,
    normalize=True,
)
if states.shape[1] != 1:
    raise RuntimeError(f"Expected one tower state in fixed-M basis, found {labels}")
scar = states[:, 0]
support = np.flatnonzero(np.abs(scar) > TOL)


def perturbation_matrix(*, distance: int, site: int):
    pairs = spin_one_xy_periodic_range_couplings(
        length=L,
        distance=distance,
        coefficient=1.0,
    )
    i, j, coeff = pairs[site]
    model = SpinOneXYChainModel(
        length=L,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=((i, j, coeff),),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    np.testing.assert_array_equal(result.basis.states, base.basis.states)
    return result.hamiltonian

odd_perturbations = [perturbation_matrix(distance=1, site=i) for i in range(L)]
even_perturbations = [perturbation_matrix(distance=2, site=i) for i in range(L)]

rows = []
for name, perturbations in (("odd-range real", odd_perturbations), ("second-neighbor real", even_perturbations)):
    report = cage_compatibility_hierarchy_from_hamiltonians(
        base.hamiltonian,
        perturbations,
        support,
        scar,
        coefficient_field="real",
        tolerance=TOL,
    )
    rows.append({
        "alphabet": name,
        "n_parameters": report.first_order.n_parameters,
        "obstruction_rank": report.first_order.rank,
        "compatible_dimension": report.first_order.compatible_dimension,
        "fixed_state_compatible_dimension": report.fixed_state.compatible_dimension,
    })

conditioning = cage_jacobian_conditioning_from_hamiltonian(
    base.hamiltonian,
    support,
    scar,
    tolerance=TOL,
)
spin_benchmark = pd.DataFrame(rows)
spin_benchmark.to_csv(DATA_DIR / "spin1_compatibility_benchmark.csv", index=False)
pd.DataFrame([conditioning.to_summary_dict()]).to_csv(DATA_DIR / "spin1_conditioning_benchmark.csv", index=False)
display(spin_benchmark)
display(pd.DataFrame([conditioning.to_summary_dict()]))

assert spin_benchmark.loc[spin_benchmark["alphabet"] == "odd-range real", "obstruction_rank"].iloc[0] == 0
assert spin_benchmark.loc[spin_benchmark["alphabet"] == "second-neighbor real", "compatible_dimension"].iloc[0] == 0


## 2. Square QDM: compact versus collective state-resolved compatibility

In [ ]:
square = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)
build = square.build(builder="sparse", backend="scipy", basis_solver="dfs", sort_basis=True)
search = CageSearcher.from_model_build_result(
    build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=TOL,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=32,
        ipr_candidate_count=16,
        ipr_random_seed=1234,
    ),
).run()
records = tuple(search[(0, 4)])
if len(records) != 9:
    raise RuntimeError(f"Expected nine (0,4) cage records, found {len(records)}")


def embedded_state(record):
    state = np.zeros(search.hilbert_size, dtype=np.complex128)
    state[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
    return state

compact = embedded_state(records[0])
collective = embedded_state(records[8])
term_builder = SparseHamiltonianBuilder(backend="scipy", dtype=np.complex128, on_missing="raise")
kinetic_terms = tuple(
    term_builder.build(build.basis, [operator]).astype(np.complex128)
    for operator in build.kinetic_operators
)

rows = []
for label, state in (("compact", compact), ("collective", collective)):
    report = operator_coefficient_compatibility(
        kinetic_terms,
        state,
        mode="fixed_vectors",
        tolerance=TOL,
    )
    rows.append({
        "target": label,
        "n_operators": report.n_operators,
        "compatible_dimension": report.compatible_dimension,
        "obstruction_rank": report.rank,
        "singular_gap": report.singular_gap,
    })
qdm_benchmark = pd.DataFrame(rows)
qdm_benchmark.to_csv(DATA_DIR / "qdm_operator_compatibility_benchmark.csv", index=False)
display(qdm_benchmark)
assert qdm_benchmark.loc[qdm_benchmark["target"] == "compact", "compatible_dimension"].iloc[0] > qdm_benchmark.loc[qdm_benchmark["target"] == "collective", "compatible_dimension"].iloc[0]


## 3. Thermal-margin utility smoke test

In [ ]:
g = np.linspace(-0.2, 0.2, 5)
activity = 0.12 - 0.05 * np.abs(g)
report = thermal_activity_margin_from_samples(
    g,
    activity,
    reference_parameter=0.0,
    tolerance=TOL,
)
summary = pd.DataFrame([report.to_summary_dict()])
summary.to_csv(DATA_DIR / "thermal_margin_smoke_test.csv", index=False)
display(summary)
assert report.reference_activity > 0.0
assert report.half_activity_radius > 0.0

fig, ax = plt.subplots(figsize=(3.35, 2.35))
ax.plot(g, activity, marker="o")
ax.axhline(report.reference_activity, linestyle="--", linewidth=0.8)
ax.set_xlabel("Path parameter")
ax.set_ylabel("Thermal activity")
ax.grid(alpha=0.3)
save_prx_figure(
    fig,
    "thermal_margin_smoke_test",
    directory=FIGURE_DIR,
    formats=("pdf", "svg"),
)
plt.show()
